In [5]:
# 1. Load the Combined Raw Data
weather_df = pd.read_csv('../data/raw/weather/weather_data/all_cities_weather.csv')

# Fixed path pointing to the true AQI file!
aqi_df = pd.read_csv('../data/raw/aqi/all_cities_aqi.csv')

print("--- WEATHER DATA ---")
print(weather_df.info())

print("--- AQI DATA ---")
print(aqi_df.info())

--- WEATHER DATA ---
<class 'pandas.DataFrame'>
RangeIndex: 3288 entries, 0 to 3287
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               3288 non-null   str    
 1   city               3288 non-null   str    
 2   temperature_max    3288 non-null   float64
 3   temperature_min    3288 non-null   float64
 4   temperature_mean   3288 non-null   float64
 5   precipitation_sum  3288 non-null   float64
 6   rain_sum           3288 non-null   float64
 7   wind_speed_max     3288 non-null   float64
dtypes: float64(6), str(2)
memory usage: 262.4 KB
None
--- AQI DATA ---
<class 'pandas.DataFrame'>
RangeIndex: 6552 entries, 0 to 6551
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    6552 non-null   str    
 1   city    6552 non-null   str    
 2   pm10    6552 non-null   float64
 3   pm25    6552 non-null   float64
dtypes: float64

In [6]:
# 2. Data Transformation: Aggregating and Merging (Updated for Open-Meteo AQI)
import os

# Step A: Standardize the Dates
# Weather dates are strings ('2026-06-03'). Convert them to actual datetime objects.
weather_df['date'] = pd.to_datetime(weather_df['date'])

# AQI dates look like '2026-06-03T14:00'. We convert to datetime and 'floor' it to the day
# so we can group it daily to match the weather.
aqi_df['date'] = pd.to_datetime(aqi_df['date']).dt.floor('D')

# Step B: Aggregate the AQI Data (No pivot needed now!)
# We group by date and city, and calculate the mean (average) for the hourly readings.
aqi_daily = aqi_df.groupby(['date', 'city'])[['pm10', 'pm25']].mean().reset_index()

print("--- AGGREGATED AQI DATA (Daily Averages) ---")
print(aqi_daily.head())

# Step C: The Grand Merge
# Combine weather and AQI on the exact same date and city
merged_df = pd.merge(weather_df, aqi_daily, on=['date', 'city'], how='inner')

print("\n--- MASTER DATASET CREATED ---")
print(merged_df.info())
print("\nMissing Values in Master:\n", merged_df.isnull().sum())

# Step D: Save to the Processed Folder
os.makedirs('../data/processed', exist_ok=True)
processed_path = '../data/processed/merged_aqi_weather.csv'
merged_df.to_csv(processed_path, index=False)

print(f"\nSaved clean data to: {processed_path}")

--- AGGREGATED AQI DATA (Daily Averages) ---
        date       city        pm10        pm25
0 2026-03-05  Bengaluru   31.875000   30.412500
1 2026-03-05      Delhi  535.279167  101.491667
2 2026-03-05  Hyderabad   37.391667   34.345833
3 2026-03-06  Bengaluru   40.804167   37.741667
4 2026-03-06      Delhi  291.304167   80.329167

--- MASTER DATASET CREATED ---
<class 'pandas.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               267 non-null    datetime64[us]
 1   city               267 non-null    str           
 2   temperature_max    267 non-null    float64       
 3   temperature_min    267 non-null    float64       
 4   temperature_mean   267 non-null    float64       
 5   precipitation_sum  267 non-null    float64       
 6   rain_sum           267 non-null    float64       
 7   wind_speed_max     267 non-null    float6